<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
%pip install -q duckdb huggingface_hub pandas numpy

In [10]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os
import json

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Connected successfully.")

Connected successfully.


In [11]:
schema = con.execute(
    f"DESCRIBE SELECT * FROM {FACT_MARCH}"
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [12]:
date_col = "report_date"
client_col = "client_hash_id"
content_col = "content_hash_id"

impressions_col = "gsc_impressions"
clicks_col = "gsc_clicks"

sessions_col = "ga4_sessions"
engaged_sessions_col = "ga4_engaged_sessions"

ga4_available_col = "ga4_data_available"

print("Columns selected successfully.")

Columns selected successfully.


In [13]:
query = f"""
SELECT
    {client_col} AS client_id,
    {content_col} AS content_id,

    -- March 1-15: information available before prediction
    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE({impressions_col}, 0)
            ELSE 0
        END
    ) AS imp_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE({clicks_col}, 0)
            ELSE 0
        END
    ) AS clicks_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN COALESCE({sessions_col}, 0)
            ELSE 0
        END
    ) AS sessions_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN COALESCE({engaged_sessions_col}, 0)
            ELSE 0
        END
    ) AS engaged_sessions_first_half,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({ga4_available_col}, FALSE)
            THEN 1
            ELSE 0
        END
    ) AS ga4_available_days,

    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND COALESCE({impressions_col}, 0) > 0
            THEN 1
            ELSE 0
        END
    ) AS active_days_first_half,

    -- March 16-31: future outcome only
    SUM(
        CASE
            WHEN {date_col} BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
            THEN COALESCE({impressions_col}, 0)
            ELSE 0
        END
    ) AS imp_second_half

FROM {FACT_MARCH}

WHERE {date_col} BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'

GROUP BY
    {client_col},
    {content_col}
"""

df = con.execute(query).df()

print("Number of pages:", len(df))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of pages: 331437


,client_id,content_id,imp_first_half,clicks_first_half,sessions_first_half,engaged_sessions_first_half,ga4_available_days,active_days_first_half,imp_second_half
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,0.0,0.0,0.0,15.0,20.0
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,0.0,0.0,0.0,13.0,403.0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,0.0,0.0,0.0,13.0,343.0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,0.0,0.0,0.0,14.0,26.0
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,0.0,0.0,0.0,14.0,1087.0


In [14]:
ga4_coverage = (
    df.groupby("ga4_available_days")
    .agg(
        n_pages=("content_id", "size")
    )
    .reset_index()
    .sort_values("ga4_available_days")
)

ga4_coverage["pct_pages"] = (
    ga4_coverage["n_pages"] / len(df) * 100
)

display(ga4_coverage)

print(
    "Pages with NO GA4 data:",
    (df["ga4_available_days"] == 0).sum()
)

print(
    "Pages with SOME GA4 data:",
    (df["ga4_available_days"] > 0).sum()
)

print(
    "Pages with FULL 15 days GA4:",
    (df["ga4_available_days"] == 15).sum()
)


,ga4_available_days,n_pages,pct_pages
0,0.0,280691,84.689096
1,1.0,22744,6.862239
2,2.0,7714,2.327441
3,3.0,4699,1.417766
4,4.0,4193,1.265097
5,5.0,2911,0.878297
6,6.0,2029,0.612183
7,7.0,1392,0.419989
8,8.0,1106,0.333698
9,9.0,912,0.275165


Pages with NO GA4 data: 280691
Pages with SOME GA4 data: 50746
Pages with FULL 15 days GA4: 335


In [15]:
print(
    "GA4 coverage rate:",
    f"{(df['ga4_available_days'] > 0).mean():.1%}"
)

GA4 coverage rate: 15.3%


In [16]:
gsc_check = con.execute(f"""
SELECT
    COUNT(DISTINCT content_hash_id) AS total_pages,

    COUNT(DISTINCT CASE
        WHEN gsc_data_available = TRUE
        THEN content_hash_id
    END) AS pages_with_gsc,

    COUNT(DISTINCT CASE
        WHEN gsc_avg_position > 0
        THEN content_hash_id
    END) AS pages_with_position

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
""").df()

display(gsc_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_pages,pages_with_gsc,pages_with_position
0,319759,151981,150675


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [17]:
position_features = con.execute(f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                 AND gsc_data_available = TRUE
            THEN 1
            ELSE 0
        END
    ) AS gsc_available_days,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
            THEN COALESCE(gsc_sum_position, 0)
            ELSE 0
        END
    ) AS sum_position_first_half

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

df = df.merge(
    position_features,
    on=["client_id", "content_id"],
    how="left"
)

df["avg_position_first_half"] = np.where(
    df["imp_first_half"] > 0,
    df["sum_position_first_half"] / df["imp_first_half"],
    np.nan
)

print("Rows:", len(df))

display(
    df[
        [
            "content_id",
            "imp_first_half",
            "clicks_first_half",
            "gsc_available_days",
            "avg_position_first_half"
        ]
    ].head(10)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331437


,content_id,imp_first_half,clicks_first_half,gsc_available_days,avg_position_first_half
0,content_39d7361b4945d504,57.0,0.0,15.0,3.964912
1,content_cec711b02f3bbde6,199.0,2.0,13.0,4.020101
2,content_275b6f7f733016d4,467.0,1.0,13.0,4.449679
3,content_ceaec531566ffcfc,56.0,0.0,14.0,7.464286
4,content_755d951187fcd70a,771.0,1.0,14.0,1.942931
5,content_7a7d3c7aa7cdfc5c,12.0,0.0,7.0,15.500000
6,content_92c381fbd361212e,233.0,0.0,13.0,5.004292
7,content_97188a7032a705cf,230.0,2.0,13.0,3.400000
8,content_c03ecafd4c999f15,4974.0,9.0,15.0,8.088460
9,content_e689bc511192751a,35.0,0.0,15.0,4.857143


In [18]:
second_half_coverage = con.execute(f"""
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(
        CASE
            WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                 AND gsc_data_available = TRUE
            THEN 1
            ELSE 0
        END
    ) AS gsc_available_days_second_half

FROM {FACT_MARCH}

WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

df = df.merge(
    second_half_coverage,
    on=["client_id", "content_id"],
    how="left"
)

df["gsc_available_days_second_half"] = (
    df["gsc_available_days_second_half"]
    .fillna(0)
)

print("Done")

display(
    df[
        [
            "content_id",
            "gsc_available_days",
            "gsc_available_days_second_half"
        ]
    ].head(10)
)

Done


,content_id,gsc_available_days,gsc_available_days_second_half
0,content_39d7361b4945d504,15.0,9.0
1,content_cec711b02f3bbde6,13.0,16.0
2,content_275b6f7f733016d4,13.0,16.0
3,content_ceaec531566ffcfc,14.0,13.0
4,content_755d951187fcd70a,14.0,16.0
5,content_7a7d3c7aa7cdfc5c,7.0,4.0
6,content_92c381fbd361212e,13.0,16.0
7,content_97188a7032a705cf,13.0,16.0
8,content_c03ecafd4c999f15,15.0,16.0
9,content_e689bc511192751a,15.0,12.0


In [19]:
print("First-half GSC coverage:")
print(df["gsc_available_days"].describe())

print("\nSecond-half GSC coverage:")
print(df["gsc_available_days_second_half"].describe())

print(
    "\nPages with full first-half coverage:",
    (df["gsc_available_days"] == 15).sum()
)

print(
    "Pages with full second-half coverage:",
    (df["gsc_available_days_second_half"] == 16).sum()
)

First-half GSC coverage:
count    319759.000000
mean          5.129604
std           6.458892
min           0.000000
25%           0.000000
50%           0.000000
75%          13.000000
max          15.000000
Name: gsc_available_days, dtype: float64

Second-half GSC coverage:
count    331437.000000
mean          5.946301
std           7.112130
min           0.000000
25%           0.000000
50%           1.000000
75%          16.000000
max          16.000000
Name: gsc_available_days_second_half, dtype: float64

Pages with full first-half coverage: 68289
Pages with full second-half coverage: 86722


In [20]:
clean_df = df[
    (df["gsc_available_days"] == 15) &
    (df["gsc_available_days_second_half"] == 16) &
    (df["imp_first_half"] > 0)
].copy()

print("All pages:", len(df))
print("Clean pages:", len(clean_df))
print(
    "Percentage kept:",
    f"{len(clean_df) / len(df):.1%}"
)

All pages: 331437
Clean pages: 61796
Percentage kept: 18.6%


In [21]:
# Average daily impressions in each time window
clean_df["avg_daily_imp_first_half"] = (
    clean_df["imp_first_half"] / 15
)

clean_df["avg_daily_imp_second_half"] = (
    clean_df["imp_second_half"] / 16
)

# Percentage change from first half to second half
clean_df["impression_change_pct"] = (
    (
        clean_df["avg_daily_imp_second_half"]
        - clean_df["avg_daily_imp_first_half"]
    )
    / clean_df["avg_daily_imp_first_half"]
) * 100

# Target:
# 1 = impressions declined by more than 20%
# 0 = did not decline by more than 20%
clean_df["is_declining_proxy"] = (
    clean_df["impression_change_pct"] < -20
).astype(int)

# CTR from March 1-15 only
clean_df["ctr_first_half"] = (
    clean_df["clicks_first_half"]
    / clean_df["imp_first_half"]
) * 100

print("Clean pages:", len(clean_df))

print(
    "Declining pages:",
    clean_df["is_declining_proxy"].sum()
)

print(
    "Declining rate:",
    f"{clean_df['is_declining_proxy'].mean():.1%}"
)

display(
    clean_df[
        [
            "content_id",
            "imp_first_half",
            "imp_second_half",
            "impression_change_pct",
            "is_declining_proxy",
            "ctr_first_half",
            "avg_position_first_half"
        ]
    ].head(10)
)

Clean pages: 61796
Declining pages: 20023
Declining rate: 32.4%


,content_id,imp_first_half,imp_second_half,impression_change_pct,is_declining_proxy,ctr_first_half,avg_position_first_half
8,content_c03ecafd4c999f15,4974.0,5875.0,10.732057,0,0.180941,8.088460
10,content_7dbc094b799e05a4,398.0,307.0,-27.685302,1,0.251256,5.298995
16,content_df22bda1218f13ff,670.0,1429.0,99.953358,0,0.000000,2.911940
27,content_b4de71c8ef5c4791,1877.0,1658.0,-17.188332,0,0.905701,2.824188
28,content_d7568011c4325a33,749.0,809.0,1.260013,0,0.133511,5.293725
30,content_e847a4dcc8af3742,969.0,1052.0,1.780186,0,0.000000,6.374613
32,content_85b1be9944e4e19d,980.0,277.0,-73.501276,1,0.000000,6.483673
36,content_b6a42d76effd1906,462.0,262.0,-46.834416,1,0.216450,18.266234
45,content_0bbda4fbddf4826f,863.0,413.0,-55.134705,1,0.000000,5.418308
47,content_32e4c08c80476c5d,82.0,110.0,25.762195,0,0.000000,20.512195


In [22]:
clean_df["volume_bucket"] = pd.qcut(
    clean_df["imp_first_half"].rank(method="first"),
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

volume_table = (
    clean_df
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        avg_impressions=("imp_first_half", "mean"),
        decline_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

volume_table["decline_rate"] = (
    volume_table["decline_rate"] * 100
)

display(volume_table)

,volume_bucket,n,avg_impressions,decline_rate
0,Low,15449,146.659266,23.347790
1,Medium,15449,423.931322,36.332449
2,High,15449,1086.102725,33.568516
3,Very High,15449,5511.078581,36.358340


### Signal 1 — Search volume

**Verdict: MIXED**

Pages with the lowest first-half search volume had the lowest observed decline rate (23.3%).
The Medium, High, and Very High buckets had higher decline rates of roughly 33.6%–36.4%, but the pattern was not monotonic.

This suggests that search volume alone is not a reliable signal of future decline in this sample.
I therefore should not treat higher volume by itself as evidence that a page is more likely to decline.

In [23]:
# Keep pages with a valid search position
ctr_position_df = clean_df[
    clean_df["avg_position_first_half"].notna()
    & (clean_df["avg_position_first_half"] > 0)
].copy()

# Create understandable Google position groups
ctr_position_df["position_bucket"] = pd.cut(
    ctr_position_df["avg_position_first_half"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "Top 3",
        "4-10",
        "11-20",
        "21-50",
        "50+"
    ],
    include_lowest=True
)

print("Pages used:", len(ctr_position_df))

display(
    ctr_position_df[
        [
            "content_id",
            "avg_position_first_half",
            "position_bucket",
            "ctr_first_half"
        ]
    ].head(10)
)

Pages used: 61795


,content_id,avg_position_first_half,position_bucket,ctr_first_half
8,content_c03ecafd4c999f15,8.088460,4-10,0.180941
10,content_7dbc094b799e05a4,5.298995,4-10,0.251256
16,content_df22bda1218f13ff,2.911940,Top 3,0.000000
27,content_b4de71c8ef5c4791,2.824188,Top 3,0.905701
28,content_d7568011c4325a33,5.293725,4-10,0.133511
30,content_e847a4dcc8af3742,6.374613,4-10,0.000000
32,content_85b1be9944e4e19d,6.483673,4-10,0.000000
36,content_b6a42d76effd1906,18.266234,11-20,0.216450
45,content_0bbda4fbddf4826f,5.418308,4-10,0.000000
47,content_32e4c08c80476c5d,20.512195,21-50,0.000000


In [24]:
# Median CTR inside each position group
ctr_position_df["position_ctr_median"] = (
    ctr_position_df
    .groupby("position_bucket", observed=True)["ctr_first_half"]
    .transform("median")
)

# If the median is above zero:
# below median = Low CTR
#
# If the median is zero:
# zero CTR = Low CTR
# positive CTR = High CTR
ctr_position_df["ctr_level"] = np.where(
    ctr_position_df["position_ctr_median"] > 0,
    np.where(
        ctr_position_df["ctr_first_half"]
        < ctr_position_df["position_ctr_median"],
        "Low CTR",
        "High CTR"
    ),
    np.where(
        ctr_position_df["ctr_first_half"] == 0,
        "Low CTR",
        "High CTR"
    )
)

In [25]:
ctr_position_table = (
    ctr_position_df
    .groupby(
        ["position_bucket", "ctr_level"],
        observed=True
    )
    .agg(
        n=("content_id", "size"),
        avg_ctr=("ctr_first_half", "mean"),
        decline_rate=("is_declining_proxy", "mean")
    )
    .reset_index()
)

ctr_position_table["decline_rate"] *= 100

display(ctr_position_table)

,position_bucket,ctr_level,n,avg_ctr,decline_rate
0,Top 3,High CTR,3777,0.649353,20.916071
1,Top 3,Low CTR,3776,0.070747,36.599576
2,4-10,High CTR,15296,0.592542,25.202667
3,4-10,Low CTR,15293,0.041607,37.899693
4,11-20,High CTR,5447,0.515441,28.235726
5,11-20,Low CTR,5446,0.000522,30.995226
6,21-50,High CTR,5609,0.300428,47.922981
7,21-50,Low CTR,5771,0.000000,33.962918
8,50+,High CTR,146,0.406912,52.054795
9,50+,Low CTR,1234,0.000000,20.259319


### Signal 2 — CTR vs Position

**Verdict: MIXED**

For pages ranking in the Top 3 and positions 4–10, low CTR was associated with a clearly higher later decline rate.

The relationship was weaker for positions 11–20 and reversed for pages ranking below position 20.

This suggests that low CTR may be a useful review signal when a page already has relatively strong search visibility, but CTR alone should not be used across all ranking positions.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue

### Baseline rule

Prioritize pages that already rank within the Top 20 but have a lower CTR than other pages with a similar search position.

Among pages that match this rule, pages with more first-half impressions are ranked higher because they represent a larger visible opportunity.

**Reason code:** `GOOD_POSITION_LOW_CTR`

**Action:** `REVIEW_CTR_OPPORTUNITY`

In [26]:
baseline_df = ctr_position_df.copy()

# Rule condition 1:
# Page is already visible in search (Top 20)
baseline_df["good_position"] = (
    baseline_df["avg_position_first_half"] <= 20
)

# Rule condition 2:
# CTR is low compared with pages in a similar position group
baseline_df["low_ctr_for_position"] = (
    baseline_df["ctr_level"] == "Low CTR"
)

# Page matches our baseline rule only if BOTH conditions are true
baseline_df["rule_match"] = (
    baseline_df["good_position"]
    & baseline_df["low_ctr_for_position"]
)

print(
    "Pages matching rule:",
    baseline_df["rule_match"].sum()
)

print(
    "Percentage matching rule:",
    f"{baseline_df['rule_match'].mean():.1%}"
)

Pages matching rule: 24515
Percentage matching rule: 39.7%


In [27]:
baseline_df["baseline_score"] = np.where(
    baseline_df["rule_match"],
    baseline_df["imp_first_half"],
    0
)

In [28]:
baseline_df["reason_code"] = np.where(
    baseline_df["rule_match"],
    "GOOD_POSITION_LOW_CTR",
    ""
)

baseline_df["action"] = np.where(
    baseline_df["rule_match"],
    "REVIEW_CTR_OPPORTUNITY",
    "MONITOR"
)

In [29]:
baseline_df = (
    baseline_df
    .sort_values(
        [
            "baseline_score",
            "avg_position_first_half"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

display(
    baseline_df[
        [
            "content_id",
            "imp_first_half",
            "avg_position_first_half",
            "ctr_first_half",
            "baseline_score",
            "reason_code",
            "action",
            "is_declining_proxy"
        ]
    ].head(10)
)

,content_id,imp_first_half,avg_position_first_half,ctr_first_half,baseline_score,reason_code,action,is_declining_proxy
0,content_7c6373141eae744a,86860.0,5.953765,0.058715,86860.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
1,content_9c057b66c30a3abb,83772.0,0.105274,0.000000,83772.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
2,content_acbcc847f8996314,83715.0,3.518354,0.158872,83715.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
3,content_34a70fea29d15f24,73639.0,2.948003,0.024444,73639.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
4,content_82e35c4845e6c391,70169.0,17.936539,0.041329,70169.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
5,content_8e1334d6356668e3,58553.0,4.753471,0.001708,58553.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
6,content_65c75874a23fca87,55680.0,6.682004,0.026940,55680.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
7,content_1642f339bd6e7c8d,52378.0,4.651991,0.034366,52378.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1
8,content_f6116743b00afc2d,49619.0,9.817671,0.016123,49619.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,0
9,content_62673eea26c31c17,49386.0,5.729114,0.070870,49386.0,GOOD_POSITION_LOW_CTR,REVIEW_CTR_OPPORTUNITY,1


In [30]:
base_rate = baseline_df["is_declining_proxy"].mean()

precision_at_10 = baseline_df.head(10)["is_declining_proxy"].mean()
precision_at_100 = baseline_df.head(100)["is_declining_proxy"].mean()

print("Base rate:", f"{base_rate:.1%}")
print("Precision@10:", f"{precision_at_10:.1%}")
print("Precision@100:", f"{precision_at_100:.1%}")

Base rate: 32.4%
Precision@10: 50.0%
Precision@100: 39.0%


### Baseline performance

- Base decline rate: **32.4%**
- Precision@10: **50.0%**
- Precision@100: **39.0%**

The rule improved Precision@100 above the overall decline base rate.
This provides a transparent baseline for the Week 5 model to beat using the same evaluation setup.

In [33]:
queue_df = baseline_df[
    baseline_df["rule_match"]
].copy()

queue_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows in action queue:", len(queue_df))
print("Saved: work/outputs/baseline_action_score.csv")

Rows in action queue: 24515
Saved: work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 10 recommendations: action, why it was selected, confidence note, and what would make the recommendation wrong.

In [34]:
top10_review = queue_df.head(10)[
    [
        "content_id",
        "imp_first_half",
        "avg_position_first_half",
        "ctr_first_half",
        "baseline_score",
        "is_declining_proxy"
    ]
].copy()

display(top10_review)

,content_id,imp_first_half,avg_position_first_half,ctr_first_half,baseline_score,is_declining_proxy
0,content_7c6373141eae744a,86860.0,5.953765,0.058715,86860.0,1
1,content_9c057b66c30a3abb,83772.0,0.105274,0.000000,83772.0,1
2,content_acbcc847f8996314,83715.0,3.518354,0.158872,83715.0,0
3,content_34a70fea29d15f24,73639.0,2.948003,0.024444,73639.0,0
4,content_82e35c4845e6c391,70169.0,17.936539,0.041329,70169.0,0
5,content_8e1334d6356668e3,58553.0,4.753471,0.001708,58553.0,0
6,content_65c75874a23fca87,55680.0,6.682004,0.026940,55680.0,1
7,content_1642f339bd6e7c8d,52378.0,4.651991,0.034366,52378.0,1
8,content_f6116743b00afc2d,49619.0,9.817671,0.016123,49619.0,0
9,content_62673eea26c31c17,49386.0,5.729114,0.070870,49386.0,1


In [35]:
top10_review["action"] = (
    "Review title/meta snippet and search intent"
)

top10_review["why"] = top10_review.apply(
    lambda row:
    f"Strong visibility ({row['imp_first_half']:.0f} impressions), "
    f"position {row['avg_position_first_half']:.2f}, "
    f"but low CTR ({row['ctr_first_half']:.3f}%).",
    axis=1
)

top10_review["confidence_note"] = (
    "Medium — the page has strong visibility and low CTR, "
    "but the CTR-vs-position signal was mixed overall."
)

top10_review["what_would_make_it_wrong"] = (
    "Low CTR may be normal for the query intent or SERP layout, "
    "so a content/snippet change may not improve performance."
)

top10_review["observed_outcome"] = np.where(
    top10_review["is_declining_proxy"] == 1,
    "Declined",
    "Did not decline"
)

display(
    top10_review[
        [
            "content_id",
            "action",
            "why",
            "confidence_note",
            "what_would_make_it_wrong",
            "observed_outcome"
        ]
    ]
)

,content_id,action,why,confidence_note,what_would_make_it_wrong,observed_outcome
0,content_7c6373141eae744a,Review title/meta snippet and search intent,"Strong visibility (86860 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Declined
1,content_9c057b66c30a3abb,Review title/meta snippet and search intent,"Strong visibility (83772 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Declined
2,content_acbcc847f8996314,Review title/meta snippet and search intent,"Strong visibility (83715 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Did not decline
3,content_34a70fea29d15f24,Review title/meta snippet and search intent,"Strong visibility (73639 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Did not decline
4,content_82e35c4845e6c391,Review title/meta snippet and search intent,"Strong visibility (70169 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Did not decline
5,content_8e1334d6356668e3,Review title/meta snippet and search intent,"Strong visibility (58553 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Did not decline
6,content_65c75874a23fca87,Review title/meta snippet and search intent,"Strong visibility (55680 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Declined
7,content_1642f339bd6e7c8d,Review title/meta snippet and search intent,"Strong visibility (52378 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Declined
8,content_f6116743b00afc2d,Review title/meta snippet and search intent,"Strong visibility (49619 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Did not decline
9,content_62673eea26c31c17,Review title/meta snippet and search intent,"Strong visibility (49386 impressions), positio...",Medium — the page has strong visibility and lo...,Low CTR may be normal for the query intent or ...,Declined


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [36]:
weak_picks = top10_review[
    top10_review["is_declining_proxy"] == 0
].copy()

print("Weak picks in Top 10:", len(weak_picks))

display(
    weak_picks[
        [
            "content_id",
            "imp_first_half",
            "avg_position_first_half",
            "ctr_first_half",
            "baseline_score"
        ]
    ]
)

Weak picks in Top 10: 5


,content_id,imp_first_half,avg_position_first_half,ctr_first_half,baseline_score
2,content_acbcc847f8996314,83715.0,3.518354,0.158872,83715.0
3,content_34a70fea29d15f24,73639.0,2.948003,0.024444,73639.0
4,content_82e35c4845e6c391,70169.0,17.936539,0.041329,70169.0
5,content_8e1334d6356668e3,58553.0,4.753471,0.001708,58553.0
8,content_f6116743b00afc2d,49619.0,9.817671,0.016123,49619.0


## 4. Weak picks and leakage

### Weak picks

Five of the Top-10 recommendations did not experience the defined later decline.

These false positives show a limitation of the baseline: strong visibility combined with low CTR does not always mean that impressions will decline later.

Possible reasons include:

- Low CTR may be normal for the query intent or SERP layout.
- The position buckets are broad and may hide differences between individual rankings.
- Search volume is used to rank matched pages by opportunity, but volume itself showed a mixed relationship with later decline.
- The baseline uses a simple rule and does not capture other factors that may affect future search performance.

### Leakage check

The baseline score uses only information available in the feature window (March 1–15, 2026):

- first-half impressions
- first-half CTR
- first-half average position

The second-half impressions and the `is_declining_proxy` target are used only after ranking to evaluate performance.

Content and client IDs are used only as identifiers, not predictive inputs.

GA4 engagement signals were not used because coverage was too limited for a fair baseline comparison.

Therefore, no future-window or label-derived feature is used to create the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ x] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [37]:
import json
import os

metrics = {
    "eligible_pages": int(len(baseline_df)),
    "rule_matches": int(baseline_df["rule_match"].sum()),
    "base_rate": float(base_rate),
    "precision_at_10": float(precision_at_10),
    "precision_at_100": float(precision_at_100),
    "volume_signal_verdict": "MIXED",
    "ctr_vs_position_signal_verdict": "MIXED"
}

os.makedirs("work/outputs", exist_ok=True)

with open(
    "work/outputs/baseline_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

{
  "eligible_pages": 61795,
  "rule_matches": 24515,
  "base_rate": 0.3240229792054373,
  "precision_at_10": 0.5,
  "precision_at_100": 0.39,
  "volume_signal_verdict": "MIXED",
  "ctr_vs_position_signal_verdict": "MIXED"
}
